# Build Person-Centric DB via SurfDrive API

It is a "file-system-based database" where separate folders are created for each person (named with personUUID). Inside each folder the full memory belonging to that person is stored, together with a `metadata.json` file 

In [1]:
import json
import os
import requests
from dotenv import load_dotenv
from pathlib import Path
from requests.auth import HTTPBasicAuth
from webdav4.client import Client

load_dotenv()

True

In [2]:
USERNAME = os.getenv("SURFDRIVE_USERNAME") # SurfDrive login email address, stored in file .env
APP_PASSWD = os.getenv("SURFDRIVE_APP_PASSWD") # create on SurfDrive at User / Settings / Security
BASE_URL = f"https://surfdrive.surf.nl/remote.php/dav/files/{USERNAME}/"

## Read Enriched JSON file (Erik)

In [3]:
FILENAME = "bhic_1921_metadata_with_scan_names.json"
DIRECTORY = "rags2riches/bhic/"

if not Path(DIRECTORY + FILENAME).exists():
    response = requests.get(BASE_URL + DIRECTORY + FILENAME, auth=HTTPBasicAuth(USERNAME, APP_PASSWD))
    response.raise_for_status()
    person_db = json.loads(response.text)
else:
    person_db = json.loads(Path(DIRECTORY / FILENAME))

person_db[0:3]

[{'identifier': '06de2958-66ce-4729-992d-6d60f940f633',
  'person_name': 'Marinus Antonius van den Kieboom',
  'death_date': '1921-11-30',
  'death_place': 'Steenbergen',
  'scan_uri': 'https://images.memorix.nl/bhic/download/fullsize/cc388f9f-d57d-ffde-e18e-e5ac2adc51df.jpg',
  'image_name': 'MFF-BergenopZoom-1918-1920-07-02954.jpg',
  'last_page': None},
 {'identifier': 'd04e75d5-f2ac-ec01-9055-8daef6a78095',
  'person_name': 'Maria Melania Celine Simons',
  'death_date': '1921-01-03',
  'death_place': 'Bergen op Zoom',
  'scan_uri': 'https://images.memorix.nl/bhic/download/fullsize/5b8b07c6-b2e4-c6cd-1b54-7db6f33b8809.jpg',
  'image_name': 'MFF-BergenopZoom-1921-1924-08-00015.jpg',
  'last_page': 'MFF-BergenopZoom-1921-1924-08-00018.jpg'},
 {'identifier': '48952bcc-7914-a45f-301f-336fdc6a0eef',
  'person_name': 'Johanna Maria Jacoba Oosterwaal',
  'death_date': '1921-01-06',
  'death_place': 'Bergen op Zoom',
  'scan_uri': 'https://images.memorix.nl/bhic/download/fullsize/d7fbc691-4

## Functions to complete the full list of image names and save them on disk

In [4]:
import re

MAIN_IMAGE_SAVE_PATH = Path("images/bhic")

def extract_info_from_image_name(regex_pattern, image_name):
    match = regex_pattern.match(image_name)
    if match:
        return match.groupdict()
    else:
        return None

def expand_bhic_filename_range(first_fname: str, last_fname: str|None) -> list[str]:

    pattern = re.compile(r"^MFF-(?P<city>[^-]+)-(?P<deel>.+)-(?P<bookid>\d+)-(?P<imageid>\d+)\.jpg$")

    # If last is None then we only know the first image anyway, so return!
    if not last_fname:
        img_info = extract_info_from_image_name(pattern, first_fname)
        return img_info['deel'], [first_fname]

    # Otherwise complete the range of all images...
    m_first = pattern.match(first_fname)
    m_last = pattern.match(last_fname)

    if not m_first or not m_last:
        raise ValueError(f"Could not parse filenames: {first_fname}, {last_fname}")

    # sanity check: everything except imageid should be identical
    for key in ("city", "deel", "bookid"):
        if m_first.group(key) != m_last.group(key):
            raise ValueError(
                f"Mismatch in '{key}': {m_first.group(key)!r} vs {m_last.group(key)!r}"
            )

    city = m_first.group("city")
    deel = m_first.group("deel")
    bookid = m_first.group("bookid")

    width = len(m_first.group("imageid"))
    start = int(m_first.group("imageid"))
    end = int(m_last.group("imageid"))

    full_image_list = [
        f"MFF-{city}-{deel}-{bookid}-{n:0{width}d}.jpg"
        for n in range(start, end + 1)
    ]

    return deel, full_image_list


def get_image_from_url(full_image_path):
    response = requests.get(full_image_path, auth=HTTPBasicAuth(USERNAME, APP_PASSWD))
    try:
        response.raise_for_status()
        return response.content
    except:
        return None

def save_image(person_id, filename, image_content):
    save_path = MAIN_IMAGE_SAVE_PATH / person_id
    save_path.mkdir(parents=True, exist_ok=True)
    file_path = save_path / filename
    with open(file_path, "wb") as f:
        f.write(image_content)

    assert Path(file_path).exists()

def save_metadata(person_obj):
    person_id = person_obj['identifier']
    json_path = MAIN_IMAGE_SAVE_PATH / person_id / "metadata.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(person_obj, f, indent=2, ensure_ascii=False)

## Iterate JSON and store only 1-page per person on disk

In [5]:
OFFICE = "Bergen op Zoom"
DIRECTORY = f"rags2riches/bhic/{OFFICE}"
n_subset = 10
first_image_only = False
for i, person_obj in enumerate(person_db[:n_subset]):
    # See progress ...
    print(f"[{i+1}/{n_subset}] Processing {person_obj['person_name']} ...")

    # Assuming this name structure only works for Brabant...?
    first_filename = person_obj.get("image_name")
    last_filename = None if first_image_only else person_obj.get("last_page")

    deel_id, images_to_process = expand_bhic_filename_range(first_filename, last_filename)

    # Get content from SURFDrive (First image only, or all images...)
    for image_name in images_to_process:
        full_image_path = f"{BASE_URL}/{DIRECTORY}/deel_{deel_id}/{image_name}"
        print(f"\t--> {image_name}")
        image_content = get_image_from_url(full_image_path)
        # Save image on disk under the personID folder (always recoverable with the json)
        person_id = person_obj['identifier']
        save_image(person_id, image_name, image_content)

    # Save metadata (to make automated evaluation possible)
    person_obj['surfdrive_path'] = f"{DIRECTORY}/deel_{deel_id}"
    person_obj['office'] = OFFICE
    person_obj['deel'] = deel_id
    save_metadata(person_obj)

[1/10] Processing Marinus Antonius van den Kieboom ...
	--> MFF-BergenopZoom-1918-1920-07-02954.jpg
[2/10] Processing Maria Melania Celine Simons ...
	--> MFF-BergenopZoom-1921-1924-08-00015.jpg
	--> MFF-BergenopZoom-1921-1924-08-00016.jpg
	--> MFF-BergenopZoom-1921-1924-08-00017.jpg
	--> MFF-BergenopZoom-1921-1924-08-00018.jpg
[3/10] Processing Johanna Maria Jacoba Oosterwaal ...
	--> MFF-BergenopZoom-1921-1924-08-00019.jpg
	--> MFF-BergenopZoom-1921-1924-08-00020.jpg
[4/10] Processing Johannes Franciscus Theuns ...
	--> MFF-BergenopZoom-1921-1924-08-00021.jpg
	--> MFF-BergenopZoom-1921-1924-08-00022.jpg
[5/10] Processing Catharina Langenberg ...
	--> MFF-BergenopZoom-1921-1924-08-00023.jpg
	--> MFF-BergenopZoom-1921-1924-08-00024.jpg
	--> MFF-BergenopZoom-1921-1924-08-00025.jpg
[6/10] Processing Pieter Schot ...
	--> MFF-BergenopZoom-1921-1924-08-00026.jpg
	--> MFF-BergenopZoom-1921-1924-08-00027.jpg
[7/10] Processing Dingena van Egeraat ...
	--> MFF-BergenopZoom-1921-1924-08-00028.j